In [1]:
# ============================================================
# PROCESS 04 - INITIAL RISK SCREENING GATEWAY (COMPLETE)
# Purpose: Screen patients BEFORE ECG upload using lifestyle/clinical data
# Model: Random Forest | XAI: SHAP | API: Flask/FastAPI Ready
# Output: 0 (Low Risk - Optional ECG) or 1 (High Risk - Mandatory ECG)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import pickle
import json
import re
import sys
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    f1_score, roc_auc_score, roc_curve, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

# Install SHAP
!pip install shap -q

print("="*70)
print("INITIAL RISK SCREENING GATEWAY - COMPLETE SYSTEM")
print("="*70)
print("API Ready for Frontend Integration\n")

# ============================================================
# 1. LOAD & PREPROCESS DATA (Pre-ECG Only)
# ============================================================

try:
    df = pd.read_csv(r"D:\coronary-ai-system\Heart_health new.csv")
    print(f"[OK] Dataset loaded: {df.shape[0]} patients")
except FileNotFoundError:
    print("[ERROR] Dataset file not found. Please check the path.")
    sys.exit(1)

# Clean duplicates
df = df.drop_duplicates()
print(f"[OK] After removing duplicates: {df.shape[0]} patients")

# Feature Engineering
df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure mmHg'].str.split('/', expand=True).astype(int)
df['Height_m'] = df['Height cm'] / 100
df['BMI'] = df['Weight kg'] / (df['Height_m'] ** 2)

# Add Lifestyle Factors
np.random.seed(42)
n_samples = len(df)

lifestyle_data = {
    'Alcohol': np.random.choice(['None', 'Light', 'Heavy'], n_samples, p=[0.5, 0.4, 0.1]),
    'Physical_Activity': np.random.choice(['Sedentary', 'Moderate', 'Active'], n_samples, p=[0.25, 0.55, 0.2]),
    'Family_History': np.random.choice([0, 1], n_samples, p=[0.75, 0.25]),
    'Diabetes': np.random.choice([0, 1], n_samples, p=[0.8, 0.2]),
    'Stress_Level': np.random.choice(['Low', 'Medium', 'High'], n_samples, p=[0.3, 0.5, 0.2]),
    'Sleep_Hours': np.clip(np.random.normal(7, 1.5, n_samples), 4, 10).round(1),
    'Years_Smoking': np.where(df['Smoker'] == 'Yes', np.random.randint(1, 35, n_samples), 0)
}

for col, vals in lifestyle_data.items():
    df[col] = vals

# Risk Flags
df['High_Cholesterol'] = (df['Cholesterol mg/dL'] > 200).astype(int)
df['High_Glucose'] = (df['Glucose mg/dL'] > 100).astype(int)
df['Hypertension'] = ((df['Systolic_BP'] >= 140) | (df['Diastolic_BP'] >= 90)).astype(int)
df['Obesity'] = (df['BMI'] >= 30).astype(int)

# Risk Score
def calc_risk_score(row):
    score = 0
    if row['Age'] > 50: score += 1
    if row['Smoker'] == 'Yes': score += 1
    if row['Diabetes'] == 1: score += 1
    if row['High_Cholesterol'] == 1: score += 1
    if row['Hypertension'] == 1: score += 1
    if row['Obesity'] == 1: score += 1
    if row['Exercise hours/week'] < 2: score += 1
    return score

df['Risk_Score'] = df.apply(calc_risk_score, axis=1)

# Encode Categoricals
encoders = {}
categorical_cols = ['Gender', 'Smoker', 'Alcohol', 'Physical_Activity', 'Stress_Level']

for col in categorical_cols:
    le = LabelEncoder()
    df[f'{col}_Encoded'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# Gateway Features
GATEWAY_FEATURES = [
    'Age', 'Gender_Encoded', 'BMI', 'Systolic_BP', 'Diastolic_BP',
    'Cholesterol mg/dL', 'Glucose mg/dL',
    'Smoker_Encoded', 'Diabetes', 'Alcohol_Encoded', 'Physical_Activity_Encoded',
    'Family_History', 'Stress_Level_Encoded', 'Sleep_Hours', 'Years_Smoking',
    'Exercise hours/week',
    'High_Cholesterol', 'High_Glucose', 'Hypertension', 'Obesity',
    'Risk_Score'
]

X = df[GATEWAY_FEATURES]
y = df['Heart Attack']

print(f"[OK] Gateway features defined: {len(GATEWAY_FEATURES)}")

# ============================================================
# 2. TRAIN MODEL WITH FULL EVALUATION
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

gateway_model = RandomForestClassifier(
    n_estimators=100, max_depth=6, min_samples_split=10,
    min_samples_leaf=4, max_features='sqrt', class_weight='balanced', random_state=42
)

gateway_model.fit(X_train_scaled, y_train)
print("[OK] Model trained successfully")

# Predictions
y_train_pred = gateway_model.predict(X_train_scaled)
y_test_pred = gateway_model.predict(X_test_scaled)
y_train_prob = gateway_model.predict_proba(X_train_scaled)[:, 1]
y_test_prob = gateway_model.predict_proba(X_test_scaled)[:, 1]

# Metrics
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
train_f1 = f1_score(y_train, y_train_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)
train_roc_auc = roc_auc_score(y_train, y_train_prob)
test_roc_auc = roc_auc_score(y_test, y_test_prob)

print("\n" + "="*70)
print("MODEL PERFORMANCE METRICS")
print("="*70)
print(f"{'Metric':<20} {'Train':>15} {'Test':>15} {'Gap':>15}")
print("-" * 70)
print(f"{'Accuracy':<20} {train_acc:>14.2%} {test_acc:>14.2%} {abs(train_acc-test_acc):>14.2%}")
print(f"{'F1-Score':<20} {train_f1:>15.4f} {test_f1:>15.4f} {abs(train_f1-test_f1):>15.4f}")
print(f"{'ROC-AUC':<20} {train_roc_auc:>15.4f} {test_roc_auc:>15.4f} {abs(train_roc_auc-test_roc_auc):>15.4f}")
print(f"{'Precision (Test)':<20} {'--':>15} {test_precision:>15.4f}")
print(f"{'Recall (Test)':<20} {'--':>15} {test_recall:>15.4f}")

# SHAP
explainer = shap.TreeExplainer(gateway_model)
print("[OK] SHAP explainer ready")

# ============================================================
# 3. INPUT VALIDATION FUNCTIONS
# ============================================================

class InputValidationError(Exception):
    """Custom exception for input validation errors"""
    pass


def validate_and_convert_input(value, field_name, expected_type, valid_options=None, min_val=None, max_val=None, pattern=None):
    """Validates and converts user input with helpful error messages"""
    
    if isinstance(value, str):
        value = value.strip()
        if value == '':
            raise InputValidationError(f"❌ {field_name} cannot be empty.")
    
    # Choice validation (case-insensitive)
    if expected_type == 'choice' and valid_options:
        value_lower = str(value).lower()
        valid_lower = [opt.lower() for opt in valid_options]
        
        if value_lower not in valid_lower:
            options_str = ', '.join([f"'{opt}'" for opt in valid_options])
            raise InputValidationError(
                f"❌ Invalid input for {field_name}: '{value}'\n"
                f"   ✓ Accepted: {options_str}"
            )
        
        idx = valid_lower.index(value_lower)
        return valid_options[idx]
    
    # Numeric validation
    if expected_type in ['int', 'float']:
        try:
            if expected_type == 'int':
                converted = int(float(value))
            else:
                converted = float(value)
        except ValueError:
            raise InputValidationError(
                f"❌ {field_name} must be a number. You entered: '{value}'"
            )
        
        if min_val is not None and converted < min_val:
            raise InputValidationError(f"❌ {field_name} too low. Min: {min_val}")
        
        if max_val is not None and converted > max_val:
            raise InputValidationError(f"❌ {field_name} too high. Max: {max_val}")
        
        return converted
    
    # Pattern validation
    if expected_type == 'pattern' and pattern:
        if not re.match(pattern, str(value)):
            raise InputValidationError(f"❌ Invalid format for {field_name}. Expected: XXX/XX")
        return value
    
    return value


def get_validated_input(prompt, field_name, expected_type, **kwargs):
    """Gets input from user with validation loop"""
    while True:
        try:
            user_input = input(prompt)
            if user_input.lower() == 'help':
                print(f"\n💡 Help for {field_name}:")
                if 'valid_options' in kwargs:
                    print(f"   Accepted values: {kwargs['valid_options']}")
                if 'min_val' in kwargs and 'max_val' in kwargs:
                    print(f"   Range: {kwargs['min_val']} - {kwargs['max_val']}")
                print()
                continue
            
            result = validate_and_convert_input(user_input, field_name, expected_type, **kwargs)
            return result
            
        except InputValidationError as e:
            print(f"\n{e}\nPlease try again...\n")


def interactive_input_with_validation():
    """Interactive questionnaire with comprehensive input validation"""
    print("\n" + "="*70)
    print("CAD RISK SCREENING - PATIENT REGISTRATION")
    print("="*70)
    print("Please answer the following questions.")
    print("Type 'help' at any prompt for guidance.\n")
    
    raw = {}
    
    # Section 1
    print("📋 SECTION 1: Basic Information")
    raw['age'] = get_validated_input("Age (years): ", "Age", 'int', min_val=18, max_val=120)
    raw['gender'] = get_validated_input("Gender (Male/Female): ", "Gender", 'choice', valid_options=['Male', 'Female'])
    
    # Section 2
    print("\n📋 SECTION 2: Body Measurements")
    raw['height_cm'] = get_validated_input("Height (cm): ", "Height", 'float', min_val=50, max_val=250)
    raw['weight_kg'] = get_validated_input("Weight (kg): ", "Weight", 'float', min_val=20, max_val=300)
    
    # Section 3
    print("\n📋 SECTION 3: Clinical Measurements")
    raw['blood_pressure'] = get_validated_input("Blood Pressure (e.g., 120/80): ", "BP", 'pattern', pattern=r'^\d{2,3}/\d{2,3}$')
    
    # Validate BP logic
    systolic, diastolic = map(int, raw['blood_pressure'].split('/'))
    while not (60 <= systolic <= 250 and 40 <= diastolic <= 150 and systolic > diastolic):
        print("❌ Invalid BP. Systolic must be > diastolic, both in normal ranges.")
        raw['blood_pressure'] = get_validated_input("Blood Pressure (e.g., 120/80): ", "BP", 'pattern', pattern=r'^\d{2,3}/\d{2,3}$')
        systolic, diastolic = map(int, raw['blood_pressure'].split('/'))
    
    raw['cholesterol'] = get_validated_input("Cholesterol (mg/dL): ", "Cholesterol", 'int', min_val=100, max_val=600)
    raw['glucose'] = get_validated_input("Glucose (mg/dL): ", "Glucose", 'int', min_val=50, max_val=600)
    
    # Section 4
    print("\n📋 SECTION 4: Lifestyle Factors")
    raw['smoker'] = get_validated_input("Do you smoke? (Yes/No): ", "Smoking", 'choice', valid_options=['Yes', 'No'])
    
    if raw['smoker'] == 'Yes':
        raw['years_smoking'] = get_validated_input("Years smoking: ", "Years", 'int', min_val=1, max_val=raw['age']-10)
    else:
        raw['years_smoking'] = 0
    
    raw['diabetes'] = get_validated_input("Diabetes? (Yes/No): ", "Diabetes", 'choice', valid_options=['Yes', 'No'])
    raw['alcohol'] = get_validated_input("Alcohol (None/Light/Heavy): ", "Alcohol", 'choice', valid_options=['None', 'Light', 'Heavy'])
    raw['physical_activity'] = get_validated_input("Activity (Sedentary/Moderate/Active): ", "Activity", 'choice', valid_options=['Sedentary', 'Moderate', 'Active'])
    raw['family_history'] = get_validated_input("Family history of CAD? (Yes/No): ", "Family History", 'choice', valid_options=['Yes', 'No'])
    raw['stress_level'] = get_validated_input("Stress Level (Low/Medium/High): ", "Stress", 'choice', valid_options=['Low', 'Medium', 'High'])
    raw['sleep_hours'] = get_validated_input("Sleep hours/night: ", "Sleep", 'float', min_val=0, max_val=24)
    raw['exercise_hours'] = get_validated_input("Exercise hours/week: ", "Exercise", 'float', min_val=0, max_val=168)
    
    print("\n✅ Registration Complete!")
    return raw


def calculate_derived_features(raw_input):
    """Calculate BMI, Risk Flags, and Risk Score"""
    height_m = raw_input['height_cm'] / 100
    bmi = raw_input['weight_kg'] / (height_m ** 2)
    systolic, diastolic = map(int, raw_input['blood_pressure'].split('/'))
    
    high_chol = 1 if raw_input['cholesterol'] > 200 else 0
    high_glucose = 1 if raw_input['glucose'] > 100 else 0
    hypertension = 1 if (systolic >= 140 or diastolic >= 90) else 0
    obesity = 1 if bmi >= 30 else 0
    
    risk_score = sum([
        raw_input['age'] > 50,
        raw_input['smoker'] == 'Yes',
        raw_input['diabetes'] == 'Yes',
        high_chol, hypertension, obesity,
        raw_input['exercise_hours'] < 2
    ])
    
    return {
        'BMI': round(bmi, 1),
        'Systolic_BP': systolic,
        'Diastolic_BP': diastolic,
        'High_Cholesterol': high_chol,
        'High_Glucose': high_glucose,
        'Hypertension': hypertension,
        'Obesity': obesity,
        'Risk_Score': risk_score
    }


def process_raw_input(raw_input):
    """Convert raw input to model-ready format"""
    derived = calculate_derived_features(raw_input)
    
    gender_encoded = encoders['Gender'].transform([raw_input['gender']])[0]
    smoker_encoded = encoders['Smoker'].transform([raw_input['smoker']])[0]
    alcohol_encoded = encoders['Alcohol'].transform([raw_input['alcohol']])[0]
    activity_encoded = encoders['Physical_Activity'].transform([raw_input['physical_activity']])[0]
    stress_encoded = encoders['Stress_Level'].transform([raw_input['stress_level']])[0]
    
    return {
        'Age': raw_input['age'],
        'Gender_Encoded': gender_encoded,
        'BMI': derived['BMI'],
        'Systolic_BP': derived['Systolic_BP'],
        'Diastolic_BP': derived['Diastolic_BP'],
        'Cholesterol mg/dL': raw_input['cholesterol'],
        'Glucose mg/dL': raw_input['glucose'],
        'Smoker_Encoded': smoker_encoded,
        'Diabetes': 1 if raw_input['diabetes'] == 'Yes' else 0,
        'Alcohol_Encoded': alcohol_encoded,
        'Physical_Activity_Encoded': activity_encoded,
        'Family_History': 1 if raw_input['family_history'] == 'Yes' else 0,
        'Stress_Level_Encoded': stress_encoded,
        'Sleep_Hours': raw_input['sleep_hours'],
        'Years_Smoking': raw_input['years_smoking'] if raw_input['smoker'] == 'Yes' else 0,
        'Exercise hours/week': raw_input['exercise_hours'],
        'High_Cholesterol': derived['High_Cholesterol'],
        'High_Glucose': derived['High_Glucose'],
        'Hypertension': derived['Hypertension'],
        'Obesity': derived['Obesity'],
        'Risk_Score': derived['Risk_Score']
    }


def screen_patient(patient_dict):
    """Main screening function"""
    # DEBUG: Print input
    print(f"\n[DEBUG] Screening patient with Risk Score: {patient_dict.get('Risk_Score')}")
    
    input_df = pd.DataFrame([patient_dict])
    X_input = input_df[GATEWAY_FEATURES]
    X_input_scaled = scaler.transform(X_input)
    
    risk_prob = gateway_model.predict_proba(X_input_scaled)[0][1]
    risk_class = gateway_model.predict(X_input_scaled)[0]
    
    print(f"[DEBUG] Predicted class: {risk_class}, Probability: {risk_prob:.4f}")
    
    # SHAP explanation
    try:
        shap_vals = explainer.shap_values(X_input_scaled)
        if isinstance(shap_vals, list):
            class_1_impacts = shap_vals[1][0]
        else:
            class_1_impacts = shap_vals[0, :, 1]
        
        contributions = [(f, class_1_impacts[i], X_input.iloc[0][f]) 
                         for i, f in enumerate(GATEWAY_FEATURES)]
        contributions.sort(key=lambda x: abs(x[1]), reverse=True)
        
        top_factors = []
        for feature, impact, value in contributions[:3]:
            effect = "higher risk" if impact > 0 else "lower risk"
            display_value = value
            if '_Encoded' in feature:
                orig = feature.replace('_Encoded', '')
                if orig in encoders:
                    try:
                        display_value = encoders[orig].inverse_transform([int(value)])[0]
                    except:
                        pass
            top_factors.append({
                'factor': feature.replace('_Encoded', '').replace('_', ' '),
                'impact': round(impact, 3),
                'value': str(display_value),
                'effect': effect
            })
    except Exception as e:
        print(f"[WARNING] SHAP failed: {e}")
        top_factors = [{'factor': 'Risk Score', 'impact': 0, 'value': str(patient_dict.get('Risk_Score')), 'effect': 'N/A'}]
    
    # Build result
    if risk_class == 1:
        result = {
            'success': True,
            'risk_status': 'HIGH_RISK',
            'risk_probability': float(risk_prob),
            'decision': 'MANDATORY_ECG',
            'message': "You show signs of elevated CAD risk based on your profile. Please upload your ECG image for detailed analysis.",
            'explanation': top_factors,
            'next_step': 'UPLOAD_ECG',
            'requires_ecg': True,
            'color': 'red'
        }
    else:
        result = {
            'success': True,
            'risk_status': 'LOW_RISK',
            'risk_probability': float(risk_prob),
            'decision': 'OPTIONAL_ECG',
            'message': "You do not currently show significant CAD risk. You may upload an ECG for additional verification if you wish.",
            'explanation': top_factors,
            'next_step': 'OPTIONAL_UPLOAD',
            'requires_ecg': False,
            'color': 'green'
        }
    
    print(f"[DEBUG] Result: {result['risk_status']}, requires_ecg: {result['requires_ecg']}")
    return result


def print_report(result):
    """Display formatted report - FIXED VERSION"""
    print("\n" + "="*70)
    print("🩺 SCREENING REPORT")
    print("="*70)
    
    # Check if result is valid
    if not isinstance(result, dict):
        print(f"❌ ERROR: Invalid result type: {type(result)}")
        return
    
    if not result.get('success', False):
        print(f"❌ ERROR: {result.get('error', 'Unknown error')}")
        print(f"Message: {result.get('message', '')}")
        return
    
    # Print main results
    print(f"Status:           {result.get('risk_status', 'N/A')}")
    print(f"Risk Probability: {result.get('risk_probability', 0):.1%}")
    print(f"Decision:         {result.get('decision', 'N/A')}")
    print("-" * 70)
    print(f"💬 Message:")
    print(f"   {result.get('message', 'No message')}")
    print("-" * 70)
    
    # Print explanation factors
    explanation = result.get('explanation', [])
    if explanation:
        print("📊 Key Contributing Factors:")
        for i, f in enumerate(explanation, 1):
            emoji = "🔴" if f.get('effect') == 'higher risk' else "🟢"
            factor_name = f.get('factor', 'Unknown')
            factor_value = f.get('value', 'N/A')
            factor_impact = f.get('impact', 0)
            print(f"   {i}. {emoji} {factor_name}: {factor_value} (impact: {factor_impact:+.3f})")
    else:
        print("📊 No explanation factors available")
    
    print("-" * 70)
    print(f"➡️  Next Step: {result.get('next_step', 'N/A')}")
    print("="*70)


# ============================================================
# 4. MAIN EXECUTION
# ============================================================

print("\n" + "="*70)
print("STARTING INTERACTIVE DEMONSTRATION")
print("="*70)

try:
    # Get input
    raw_patient = interactive_input_with_validation()
    print(f"\n[OK] Collected data for age {raw_patient['age']} {raw_patient['gender']}")
    
    # Process
    processed = process_raw_input(raw_patient)
    print(f"[OK] Processed data - BMI: {processed['BMI']}, Risk Score: {processed['Risk_Score']}")
    
    # Screen
    result = screen_patient(processed)
    
    # Print report
    print_report(result)
    
except Exception as e:
    print(f"\n❌ FATAL ERROR: {str(e)}")
    import traceback
    traceback.print_exc()

# ============================================================
# 5. SAVE MODEL ARTIFACTS
# ============================================================
print("\n" + "="*70)
print("SAVING MODEL ARTIFACTS")
print("="*70)

try:
    pickle.dump(gateway_model, open('gateway_model.pkl', 'wb'))
    pickle.dump(scaler, open('gateway_scaler.pkl', 'wb'))
    pickle.dump(encoders, open('gateway_encoders.pkl', 'wb'))
    pickle.dump(GATEWAY_FEATURES, open('gateway_features.pkl', 'wb'))
    
    metrics_dict = {
        'train_accuracy': train_acc, 'test_accuracy': test_acc,
        'train_f1': train_f1, 'test_f1': test_f1,
        'train_roc_auc': train_roc_auc, 'test_roc_auc': test_roc_auc,
        'test_precision': test_precision, 'test_recall': test_recall
    }
    pickle.dump(metrics_dict, open('model_metrics.pkl', 'wb'))
    
    print("[OK] All artifacts saved successfully")
    print(f"   - gateway_model.pkl")
    print(f"   - gateway_scaler.pkl")
    print(f"   - gateway_encoders.pkl")
    print(f"   - gateway_features.pkl")
    print(f"   - model_metrics.pkl")
    
except Exception as e:
    print(f"[ERROR] Failed to save artifacts: {e}")

print("\n" + "="*70)
print("SYSTEM READY")
print("="*70)

C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Users\HP\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


INITIAL RISK SCREENING GATEWAY - COMPLETE SYSTEM
API Ready for Frontend Integration

[OK] Dataset loaded: 724 patients
[OK] After removing duplicates: 714 patients
[OK] Gateway features defined: 21
[OK] Model trained successfully

MODEL PERFORMANCE METRICS
Metric                         Train            Test             Gap
----------------------------------------------------------------------
Accuracy                     99.25%         98.88%          0.37%
F1-Score                      0.9937          0.9906          0.0032
ROC-AUC                       0.9999          0.9997          0.0002
Precision (Test)                  --          1.0000
Recall (Test)                     --          0.9813
[OK] SHAP explainer ready

STARTING INTERACTIVE DEMONSTRATION

CAD RISK SCREENING - PATIENT REGISTRATION
Please answer the following questions.
Type 'help' at any prompt for guidance.

📋 SECTION 1: Basic Information


KeyboardInterrupt: Interrupted by user